# 04.1 · q-transfer (Spec-1) — data pipeline demo

`SRC.task == DST.task`, `SRC.progress != DST.progress`. One multi-predicate task; a different subset is prefilled as done in SRC vs DST, so at the shared pivot they are about to write different next sub-goals. Transplant the SRC pivot **query** into DST → does DST now pursue SRC's next sub-goal (`expected_next`)?

This notebook demonstrates `src.diff.data.q_transfer_setups` + the query-inject machinery from `src.diff.functional`.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-09-02 15:22:03 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Inspect the generated contrastive setups

In [2]:
# a few contrastive q-transfer setups (one datatype, listed presentation, python)
setups = D.q_transfer_setups("list[str]", presentation="listed", src_lang="python", dst_lang="python")
print(f"{len(setups)} setups for list[str]:")
for s in setups:
    print(f"  expected_next={s.expected_next:<18} src_progress={s.src_progress}  dst_progress={s.dst_progress}")

# datatype x presentation x language coverage (string render only — no model needed)
for dt in D.QTASKS:
    for pres in ("listed", "description"):
        n = len(D.q_transfer_setups(dt, presentation=pres))
        print(f"  {dt:<10} {pres:<12} -> {n} setups")

4 setups for list[str]:
  expected_next=count_long         src_progress=['has_palindrome', 'total_length']  dst_progress=['has_palindrome']
  expected_next=count_capitalized  src_progress=['has_palindrome', 'total_length', 'count_long']  dst_progress=['has_palindrome']
  expected_next=distinct_count     src_progress=['has_palindrome', 'total_length', 'count_long', 'count_capitalized']  dst_progress=['has_palindrome']
  expected_next=count_capitalized  src_progress=['has_palindrome', 'total_length', 'count_long']  dst_progress=['has_palindrome', 'total_length']
  list[str]  listed       -> 4 setups
  list[str]  description  -> 4 setups
  list[int]  listed       -> 4 setups
  list[int]  description  -> 4 setups
  int        listed       -> 4 setups
  int        description  -> 4 setups
  str        listed       -> 4 setups
  str        description  -> 4 setups


## Render one setup — SRC and DST prompts with pivots

In [3]:
inst = D.q_transfer_setups(
    "list[str]", 
    presentation="description", 
    src_lang="python", dst_lang="cpp", 
    n_conditions=3,
    naming="letters"
)[0]
m = D.materialize(inst)                       # applies the chat template
src_prompt, dst_prompt = m["src_prompt"], m["dst_prompt"]
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
print(f"expected_next = {inst.expected_next!r}  ({inst.expected_next_desc})")
show(src_prompt, f"SRC · progress={inst.src_progress} · about to write {inst.expected_next!r} · pivot@{piv_src}", piv=-1)
show(dst_prompt, f"DST · progress={inst.dst_progress} · pivot@{piv_dst}", piv=-1)

expected_next = 'count_long'  (the number of words longer than five characters)


╭─ SRC · progress=['has_palindrome', 'total_length'] · about to write 'count_long' · pivot@130 ───────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ whether any word is a palindrome; the total number of characters across all words; the number of words longer   │
│ than five characters\n                                                                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = any(w == w[::-1] for w in words)\n                                                                      │
│     b = sum(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · progress=['has_palindrome'] · pivot@162 ─────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ whether any word is a palindrome; the total number of characters across all words; the number of words longer   │
│ than five characters\n                                                                                          │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     bool a = false;\n                                                                                           │
│     for (auto& w : words) { std::string rv(w.rbegin(), w.rend()); if (w == rv) { a = true; break; } }\n         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant the SRC pivot query into DST

Cache the (pre-RoPE) query of `PATCH_HEADS` at the SRC pivot; inject it at the DST pivot (prefill + re-clamped for `n_steer` decode steps). Expect the patched DST to implement `expected_next` instead of its own next predicate.

In [4]:
# ── intervention: transplant the SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
q_src = cache_src_proj(src_prompt, [piv_src], Q_LAYERS, "q")

def q_inject_specs(qcache, dest_pivot, heads=PATCH_HEADS):
    qh = heads_per_layer(heads, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],   # prefill
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])            # decode re-clamp

src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)
pf, dc  = q_inject_specs(q_src, piv_dst)

# clean baseline attn over the TASK (numbered predicate list); query = pivot (last token)
_spans    = [token_span(dst_prompt, p.description) for p in inst.task.predicates]
TASK_POS  = list(range(min(s[0] for s in _spans), max(s[1] for s in _spans)))
task_toks = [mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(dst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, piv_dst)

# ── source_run — clean SRC (frontier = expected_next); generation + goalscope ──
gen_src = greedy(src_tok, max_new=96)
rich_print(f"[red1]source_run · clean SRC (frontier {inst.expected_next!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_prompt, piv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); generation + goalscope + ATTN_VIS baseline ──
gen_dst = greedy(dst_tok, max_new=96)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {inst.meta['dst_next']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_prompt, piv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
attn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); gen + goalscope + ATTN_DIFF ──
_iface = make_patch_interface(mt, pf)
gen_patched = greedy(dst_tok, prefill_specs=pf, decode_specs=dc, n_steer=16, max_new=96)
rich_print(f"[salmon1]patched_run · DST (expect {inst.expected_next!r}):[/salmon1] \n{escape(gen_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(dst_tok, mt, value_weighted=True, patch_interface=_iface)
attn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, piv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, piv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, attn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'count_long'): 
 c = sum(1 for w in words if len(w) > 5)
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source_run  pos=130 '    ' => I'm counting lines where the length is greater than 5.

destination_run · clean DST (frontier 'total_length'): 
 int b = 0;
    for (auto& w : words) { b += w.size(); }
    int c = 0;
    for (auto& w : words) { if (w.size() > 5) { c++; } }
    return std::make_tuple(a, b, c);
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=162 '    ' => I'm counting how many lines have more than 5 characters.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=0.638)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=0.379)

---
## Evaluate the intervention (`src.diff.eval`)

Three complementary evals of the q-transfer intervention (single source of truth in `src/diff/eval.py`):

1. **Causal** (what we care about) — oracle judges whether the patched continuation pursues the SRC target (or a non-trivial part).
2. **Readout shift** — classifies all three goalscope readouts (source/destination/patched); a case is *valid* only if the clean readouts land as expected (source→SRC, dest→DST), and *shifted* if the patched readout expresses the SRC target.
3. **Gaze** — rule-based: did the value-weighted attention delta move toward the SRC-target tokens vs the DST-target tokens?

The causal & readout evals call the Claude oracle (`ask_claude`, disk-cached). The same `eval_q_transfer` works on the composed-filter instance below.

In [5]:
from src.diff import eval as E

qeval = E.eval_q_transfer(
    inst,
    patched_gen=gen_patched,                                   # 1. causal (oracle)
    readouts=(ro_src, ro_dst, ro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=dst_prompt, task_positions=TASK_POS,     # 3. gaze (rule-based)
              attn_delta=attn_patched_goal - attn_clean_goal),
)
rich_print(f"[bold]EVAL[/bold] · expected_next={inst.expected_next!r}\n{qeval.summary()}")
rich_print("  causal :", qeval.causal)
rich_print("  readout:", qeval.readout)
rich_print("  gaze   :", qeval.gaze)

EVAL · expected_next='count_long'
causal: TRIGGERED (both) | readout: src→neither→src [INVALID, shifted] | gaze: src+0.178 vs dst-0.109 (shifted)

causal :
{
    'label': 'both',
    'why': 'The code computes both the count of words longer than five characters (variable `b` using `count_if`) 
and the total number of characters across all words (variable `c` using `accumulate`).',
    'raw': '<verdict>BOTH</verdict>\n<why>The code computes both the count of words longer than five characters 
(variable `b` using `count_if`) and the total number of characters across all words (variable `c` using 
`accumulate`).</why>',
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'neither',
    'patched': 'src',
    'valid': False,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase explicitly mentions counting lines/words where the length is greater than 5, which 
directly corresponds to sub-goal (A).',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly mentions counting lines/words where the length
is greater than 5, which directly corresponds to sub-goal (A).</why>'
        },
        'dest': {
            'label': 'neither',
            'why': 'The phrase is too generic ("counting characters and checking conditions") to clearly indicate 
either counting total characters (B) or counting words longer than five characters (A) specifically.',
            'raw': '<verdict>NEITHER</verdict>\n<why>The phrase is too generic ("counting characters and checking 
conditions") to clearly indicate either counting total characters (B) or counting words longer than five characters
(A) specifically.</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'The phrase explicitly mentions counting lines/words with more than 5 characters, which 
corresponds to sub-goal (A).',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly mentions counting lines/words with more than 5
characters, which corresponds to sub-goal (A).</why>'
        }
    }
}

gaze   :
{
    'src_delta': 0.1783077623695135,
    'dst_delta': -0.10862101055681705,
    'shifted': True,
    'margin': 0.28692877292633057
}

---
## Composed-filter variant (`filter_transfer_setups`) — mid-expression pivot

Same query-transfer *mechanism*, but the sub-goals are **commutative conditions composed inside one return expression** (the original 00_diff TASK_1 framing: `return all the words that are palindromes and start with a vowel`). The pivot sits **mid-expression** — right after `… if <cond> and` — so the model completes the *same* comprehension by adding the next condition, rather than opening a fresh assignment statement.

The intervention is unchanged (reuses `q_inject_specs`); only the ATTN **TASK span** is located differently — the composed phrase `m["task_text"]`, not a per-predicate numbered list (`FilterTask` has `.conditions`, not `.predicates`).

In [6]:
# composed-filter setups: 2 commutative sub-tasks; SRC/DST prefill DIFFERENT one (00_diff TASK_1).
fsetups = D.filter_transfer_setups("list[str]")          # postfix="targeted" by default
print(f"{len(fsetups)} filter setups for list[str] — TASK: {fsetups[0].task_text!r}")
for s in fsetups:
    print(f"  src_fill={s.src_fill:<12} dst_fill={s.dst_fill:<12} -> expected_next={s.expected_next!r} "
          f"(clean DST -> {s.meta['dst_next']!r})")
print()
for dt in D.FILTER_TASKS:
    print(f"  {dt:<10} -> {len(D.filter_transfer_setups(dt))} setups ({len(D.FILTER_TASKS[dt].conditions)} sub-tasks in bank)")

4 filter setups for list[str] — TASK: 'all the words in `words` that are palindromes and start with a vowel'
  src_fill=palindrome   dst_fill=vowel_start  -> expected_next='vowel_start' (clean DST -> 'palindrome')
  src_fill=vowel_start  dst_fill=palindrome   -> expected_next='palindrome' (clean DST -> 'vowel_start')
  src_fill=palindrome   dst_fill=long         -> expected_next='long' (clean DST -> 'palindrome')
  src_fill=long         dst_fill=palindrome   -> expected_next='palindrome' (clean DST -> 'long')

  list[str]  -> 4 setups (3 sub-tasks in bank)
  list[int]  -> 4 setups (3 sub-tasks in bank)
  str        -> 4 setups (3 sub-tasks in bank)


In [7]:
finst = D.filter_transfer_setups(
    "list[str]",
    src_lang="python", dst_lang="python",
    postfix="targeted"
)[0]          # or postfix="return" for the loose pivot
fm = D.materialize(finst)
fsrc, fdst = fm["src_prompt"].rstrip(), fm["dst_prompt"].rstrip()
fpiv_src, fpiv_dst = pivot_of(fsrc), pivot_of(fdst)
print(f"expected_next = {finst.expected_next!r}  ({finst.expected_next_desc})")
show(fsrc, f"SRC · prefilled {finst.src_fill!r} · about to apply {finst.expected_next!r} · pivot@{fpiv_src}",
     emph_span=token_span(fsrc, fm["task_text"]), piv=-1)
show(fdst, f"DST · prefilled {finst.dst_fill!r} · about to apply {finst.meta['dst_next']!r} · pivot@{fpiv_dst}",
     emph_span=token_span(fdst, fm["task_text"]), piv=-1)

expected_next = 'vowel_start'  (start with a vowel)


╭─ SRC · prefilled 'palindrome' · about to apply 'vowel_start' · pivot@121 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that are palindromes and start with a vowel\n                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w == w[::-1]]\n                                                                  │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · prefilled 'vowel_start' · about to apply 'palindrome' · pivot@126 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that are palindromes and start with a vowel\n                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w[:1].lower() in 'aeiou']\n                                                      │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Intervention — same query transplant, TASK span = the composed phrase

Reuses `q_inject_specs` / `Q_LAYERS` from the q-transfer intervention above; the only change is `TASK_POS = token_span(fdst, m["task_text"])`.

In [8]:
PATCH_HEADS = GOAL_HEADS  # heads carrying the transplant (GOAL | UNION | ...)
len(PATCH_HEADS)

122

In [9]:
# ── composed-filter intervention: transplant SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS  = sorted({l for l, _ in PATCH_HEADS})
fq_src    = cache_src_proj(fsrc, [fpiv_src], Q_LAYERS, "q")
fsrc_tok  = prepare_input(prompts=fsrc, tokenizer=mt)
fdst_tok  = prepare_input(prompts=fdst, tokenizer=mt)
fpf, fdc  = q_inject_specs(fq_src, fpiv_dst)              # SAME machinery as the q-transfer cell

# TASK span = the single COMPOSED phrase (FilterTask has .conditions, not a per-line list); query = pivot
TASK_POS  = list(range(*token_span(fdst, fm["task_text"])))
task_toks = [mt.tokenizer.decode([int(fdst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(fdst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, fpiv_dst)

# ── source_run — clean SRC (frontier = expected_next) ──
fgen_src = greedy(fsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC (frontier {finst.expected_next!r}):[/red1] \n{escape(fgen_src)}")
fro_src = goalscope_readout("   ↳ goalscope · source_run", fsrc, fpiv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); ATTN_VIS baseline ──
fgen_dst = greedy(fdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {finst.meta['dst_next']!r}):[/sea_green3] \n{escape(fgen_dst)}")
fro_dst = goalscope_readout("   ↳ goalscope · destination_run", fdst, fpiv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
fattn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); ATTN_DIFF ──
_iface = make_patch_interface(mt, fpf)
fgen_patched = greedy(fdst_tok, prefill_specs=fpf, decode_specs=fdc, n_steer=32, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {finst.expected_next!r}):[/salmon1] \n{escape(fgen_patched)}")
fro_patched = goalscope_readout("   ↳ goalscope · patched_run", fdst, fpiv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(fdst_tok, mt, value_weighted=True, patch_interface=_iface)
fattn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, fpiv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, fpiv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, fattn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'vowel_start'): 
 w[0].lower() in 'aeiou']
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=121 ' if' => I'm checking if the character is a vowel (a, e, i, o, u) and also if 
it's lowercase.

destination_run · clean DST (frontier 'palindrome'): 
 w == w[::-1]]
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=126 ' if' => I'm checking if the string is equal to its reverse.

... (2 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=126 ' if' => I'm checking if the character is a vowel.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.234)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=1.590)

### Evaluate the composed-filter intervention

Same `eval_q_transfer` (the filter instance exposes the same `expected_next_desc` / `dst_next_desc` / `task_desc`), on the filter run's captured outputs.

In [10]:
feval = E.eval_q_transfer(
    finst,
    patched_gen=fgen_patched,                                    # 1. causal (oracle)
    readouts=(fro_src, fro_dst, fro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=fdst, task_positions=TASK_POS,              # 3. gaze (rule-based)
              attn_delta=fattn_patched_goal - fattn_clean_goal),
)
rich_print(f"[bold]EVAL (filter)[/bold] · expected_next={finst.expected_next!r}\n{feval.summary()}")
rich_print("  causal :", feval.causal)
rich_print("  readout:", feval.readout)
rich_print("  gaze   :", feval.gaze)

EVAL (filter) · expected_next='vowel_start'
causal: TRIGGERED (src) | readout: src→dst→src  | gaze: src+0.350 vs dst-0.338 (shifted)

causal :
{
    'label': 'src',
    'why': "The code checks if a word starts with a vowel using `startswith(('a', 'e', 'i', 'o', 'u'))` and 
redundantly checks `w.lower()[0] in 'aeiou'`, but contains no palindrome check.",
    'raw': "<verdict>A</verdict>\n<why>The code checks if a word starts with a vowel using `startswith(('a', 'e', 
'i', 'o', 'u'))` and redundantly checks `w.lower()[0] in 'aeiou'`, but contains no palindrome check.</why>",
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase explicitly discusses checking if a character is a vowel (a, e, i, o, u) and 
lowercase, which directly pursues sub-goal (A): start with a vowel.',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly discusses checking if a character is a vowel 
(a, e, i, o, u) and lowercase, which directly pursues sub-goal (A): start with a vowel.</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'Checking if a string equals its reverse is the classic palindrome check, directly pursuing 
sub-goal (B).',
            'raw': '<verdict>B</verdict>\n<why>Checking if a string equals its reverse is the classic palindrome 
check, directly pursuing sub-goal (B).</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'The phrase explicitly mentions checking if a character is a vowel, which directly pursues 
sub-goal (A) about starting with a vowel.',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly mentions checking if a character is a vowel, 
which directly pursues sub-goal (A) about starting with a vowel.</why>'
        }
    }
}

gaze   :
{'src_delta': 0.3501648996025324, 'dst_delta': -0.33792045898735523, 'shifted': True, 'margin': 0.6880853585898876}